<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/social_media/Final_tweet_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kwanda Mazibuko** - stdnr: 1077167

# **Importing Libraries**

In [1]:
!pip install -q tweepy gensim python-louvain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.3 MB/s eta 0:00:00


In [2]:
# Importing Libraries - make sure the packages are installed
import os
import tweepy as tw
import re
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO

import networkx as nx
import community as community_louvain
from community.community_louvain import best_partition

import warnings
warnings.filterwarnings("ignore")

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# **Loading Data**

In [3]:
# Reading data
%%time
FILE_ID = "1lhoPhOLB4fm-IgXlKhZFfC0UiFfHSklZ"
xlsx_url = f"https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx"
r = requests.get(xlsx_url)
df_1 = pd.read_excel(BytesIO(r.content), engine = "openpyxl")


CPU times: user 1min 41s, sys: 707 ms, total: 1min 42s
Wall time: 1min 46s


In [4]:
df_1.head(1)

,Query Id,Query Name,Date,Title,Url,Domain,Sentiment,Page Type,Language,Country Code,Continent Code,Continent,Country,City Code,Account Type,Added,Assignment,Author,Category Details,Checked,City,Display URLs,Entity Info,Expanded URLs,Facebook Author ID,Facebook Comments,Facebook Likes,Facebook Role,Facebook Shares,Facebook Subtype,Full Name,Full Text,Gender,Impressions,Instagram Comments,Instagram Followers,Instagram Following,Instagram Interactions Count,Instagram Posts,Interest,Last Assignment Date,Latitude,Location Name,Longitude,Media Filter,Media URLs,Mentioned Authors,Original Url,Priority,Professions,Resource Id,Short URLs,Starred,Station Name,Viewership,Status,Subtype,Thread Author,Thread Created Date,Thread Entry Type,Thread Id,Thread URL,Total Monthly Visitors,X Author ID,X Channel Role,X Followers,X Following,X Replies,X Reply to,X Repost of,X Reposts,X Likes,X Posts,X Verified,Updated,Reach (new),Publication Name,Licenses,Redacted,Redacted Fields,Redaction Reason,Asset Content Id,Asset Thumb Id,Author Verified Type,Avatar,Batch Id,Blog Name,Broadcast Media Url,Is Syndicated,Air Type,Broadcast Type,Media Type,Ad Value,Circulation,Region,Region Code,Daily Visitors,Engagement Type,Hashtags,Item Review,Kicker,Linkedin Comments,Linkedin Engagement,Linkedin Impressions,Linkedin Likes,Linkedin Shares,Linkedin Sponsored,Linkedin Video Views,Parent Post Id,Parent Blog Name,Pub Type,Publisher Sub Type,Rating,Reddit Score,Reddit Score Upvote Ratio,Reddit Comments,Reddit Author Karma,Root Post Id,Root Blog Name,Subreddit,Subreddit Subscribers,Subscriptions,Sub Title,React Score Overall,React Score Emotionality,React Score Harmful,Engagement Score,Subreddit NSFW,Reddit Post Flair,Reddit Author Flair,Subreddit Topics,Reddit Spoiler,Publication Id,Page Type Name,Content Source,Content Source Name,Custom,Bluesky Author Id,Bluesky Followers,Bluesky Following,Bluesky Likes,Bluesky Posts,Bluesky Quotes,Bluesky Replies,Bluesky Reposts,Can Edit Markup,Can Edit Metadata,Can Edit Segmentation,Can Edit Workflow,Copyright,Factiva Attribute Code,Has Full Text,Impact,Instagram Likes,Mention Id,Podcast Audience Estimate,Podcast Duration Ms,Raw Metadata,Reportable,Threads Likes,Threads Quotes,Threads Replies,Threads Reposts,Threads Shares,Threads Views,Tiktok Comments,Tiktok Connected Account,Tiktok Likes,Tiktok Reach,Tiktok Shares,Tiktok Views,Weblog Title,Youtube Comments,Youtube Duration Milliseconds,Youtube Favourites,Youtube Likes,Youtube Subscriber Count,Youtube Video Count,Emotion
0,2003594270,Kenya protests 2025,2025-08-31 21:59:50.0,RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no escape route for the wicked!\n\n#DualeMustGo #RutoMustGo #DrainTheSwamp,http://twitter.com/kelvinngari62/statuses/1962274155270635732,twitter.com,negative,twitter,en,KEN,AFRICA,Africa,Kenya,KEN.Coast.Mombasa,individual,2025-09-02T09:16:47.214+0000,NaN,kelvinngari62,NaN,False,Mombasa,NaN,"{entityId=13414952, entityConfidence=HIGH, url=https://www.wikidata.org/wiki/Q13414952}, {entityId=43169, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q43169}, {entityId=2727213, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q2727213}, {entityId=4682154, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q4682154}, {entityId=497, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q497}",NaN,NaN,0,0,NaN,0,NaN,kelvinngari62 (knn),RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no e

In [5]:
df_1.shape

(49823, 179)

#### **Pre-Processing**

In [6]:
# Changing column names
df_1.columns = df_1.columns.str.replace(' ', '_', regex=False).str.lower()

In [7]:
# Adding datetime
df_1['date'] = pd.to_datetime(df_1['date'])
df_june_2025 = df_1[(df_1['date'].dt.month == 6) & (df_1['date'].dt.year == 2025)]

# **Question One**





#### **a. (i) User-Mention Network**

**Nodes** will be authors and mentioned users, and **edges** will go from an author to a mentioned user.


In [10]:
# Function to extract mentions from full_text (re-included from cell 297c898e)
def extract_mentions(text):
    mentions = re.findall(r'@(\w+)', str(text))
    return list(set(mentions))

# Add mentions and tweet_author
df_june_2025['tweet_author'] = df_june_2025['author']
df_june_2025['mentions'] = df_june_2025['full_text'].apply(extract_mentions)

# Add graph
G_mentions = nx.DiGraph()
for index, row in df_june_2025.iterrows():
    author = row['tweet_author']

    mentions = row['mentions']
    if author not in G_mentions:
        G_mentions.add_node(author)

    if mentions:
        for mentioned_user in mentions:
            if mentioned_user not in G_mentions:
                G_mentions.add_node(mentioned_user)
            G_mentions.add_edge(author, mentioned_user)

print(f"Number of nodes in G_mentions: {G_mentions.number_of_nodes()}")
print(f"Number of edges in G_mentions: {G_mentions.number_of_edges()}")

Number of nodes in G_mentions: 14452
Number of edges in G_mentions: 33555


**User-Mention Network Metrics**

 - Calculate degree centrality, betweenness centrality, and clustering coefficient.

In [13]:
%%time
# Calculate Degree Centrality
degree_centrality = nx.degree_centrality(G_mentions)
print('Top 2 Degree Centrality')
for node, centrality in sorted(degree_centrality.items(), key = lambda item: item[1], reverse = True)[:2]:
    print(f"Node: {node}, Degree Centrality: {centrality:.4f}")

print('')
# Calculate Betweenness
betweenness_centrality = nx.betweenness_centrality(G_mentions)
print('Top 2 Betweenness')
for node, centrality in sorted(betweenness_centrality.items(), key = lambda item: item[1], reverse = True)[:2]:
    print(f"Node: {node}, Betweenness: {centrality:.4f}")

print('')
# Calculate Clustering Coefficient
clustering_coefficient = nx.clustering(G_mentions)
print('Top 2 Clustering Coefficient')

# Filter out nodes with 0 clustering coefficient
meaningful_clustering = {node: coeff for node, coeff in clustering_coefficient.items() if coeff > 0}
for node, coeff in sorted(meaningful_clustering.items(), key = lambda item: item[1], reverse = True)[:2]:
    print(f"Node: {node}, Clustering Coefficient: {coeff:.4f}")


Top 2 Degree Centrality
Node: IAMRAPCHA, Degree Centrality: 0.0540
Node: FGaitho237, Degree Centrality: 0.0453

Top 2 Betweenness
Node: C_NyaKundiH, Betweenness: 0.0005
Node: JaokooMoses, Betweenness: 0.0004

Top 2 Clustering Coefficient
Node: kagiajames2, Clustering Coefficient: 0.5000
Node: Farida_N, Clustering Coefficient: 0.5000
CPU times: user 7min 56s, sys: 402 ms, total: 7min 56s
Wall time: 7min 59s


#### Top 15 Influencial Nodes

In [ ]:
print('')
print('Top 15 Influential Nodes by Degree Centrality')
# Sort degree_centrality in descending order and print top 15
sorted_degree_centrality = sorted(degree_centrality.items(), key = lambda item: item[1], reverse = True)
for node, centrality in sorted_degree_centrality[:15]:
    print(f"Node: {node}, Degree Centrality: {centrality:.4f}")

print('')
print('Top 15 Influential Nodes by Betweenness')
# Sort betweenness_centrality in descending order and print top 15
sorted_betweenness_centrality = sorted(betweenness_centrality.items(), key = lambda item: item[1], reverse = True)
for node, centrality in sorted_betweenness_centrality[:15]:
    print(f"Node: {node}, Betweenness Centrality: {centrality:.4f}")

**User-Mention Network for Gephi**

In [15]:
# Add calculated metrics as node attributes to G_mentions (full graph)
for node, centrality in degree_centrality.items():
    G_mentions.nodes[node]['degree_centrality'] = centrality

for node, centrality in betweenness_centrality.items():
    G_mentions.nodes[node]['betweenness_centrality'] = centrality

for node, coeff in clustering_coefficient.items():
    G_mentions.nodes[node]['clustering_coefficient'] = coeff

# Identify top influential nodes for user-mention network
top_degree_nodes = {node for node, centrality in sorted_degree_centrality[:15]}
top_betweenness_nodes = {node for node, centrality in sorted_betweenness_centrality[:15]}

# Combine these sets to get all unique top influential nodes
influential_nodes_mentions = list(top_degree_nodes.union(top_betweenness_nodes))

# Create a subgraph with only these influential nodes and their connections
G_mentions_influential = G_mentions.subgraph(influential_nodes_mentions)

In [16]:
# Save the G_mentions_influential graph to a GEXF file
nx.write_gexf(G_mentions_influential, 'user_mention_network_top_influencers.gexf')
print('export successful')

export successful


#### **a. (ii) Retweet Network**

**Nodes** will be the authors of the original tweets and the authors who retweeted them, and **edges** will go from the retweeting user to the retweeted user.


In [17]:
# Function to extract retweeted author
def extract_retweet_author(text):
    retweet_match = re.match(r'RT @(\w+)', str(text))
    if retweet_match:
        return retweet_match.group(1)
    return None

# Adding 'retweeted_author' column
df_june_2025['retweeted_author'] = df_june_2025['full_text'].apply(extract_retweet_author)

# Graph
G_retweets = nx.DiGraph()
for index, row in df_june_2025.iterrows():
    retweeter = row['tweet_author']
    retweeted = row['retweeted_author']

    if retweeter not in G_retweets:
        G_retweets.add_node(retweeter)

    if retweeted:
        if retweeted not in G_retweets:
            G_retweets.add_node(retweeted)
        G_retweets.add_edge(retweeter, retweeted)

# 6. Print the total number of nodes and edges
print(f"Number of nodes in G_retweets: {G_retweets.number_of_nodes()}")
print(f"Number of edges in G_retweets: {G_retweets.number_of_edges()}")

Number of nodes in G_retweets: 13875
Number of edges in G_retweets: 28869


**User-Mention Network Metrics**

 - Calculate degree centrality, betweenness centrality, and clustering coefficient.

In [19]:
%%time
# Calculate Degree Centrality
degree_centrality_retweets = nx.degree_centrality(G_retweets)
print('Top 2 Degree Centrality (Retweets)')
for node, centrality in sorted(degree_centrality_retweets.items(), key = lambda item: item[1], reverse = True)[:2]:
    print(f"Node: {node}, Degree Centrality: {centrality:.4f}")

print('')
# Calculate Betweenness
betweenness_centrality_retweets = nx.betweenness_centrality(G_retweets)
print('Top 2 Betweenness (Retweets)')
for node, centrality in sorted(betweenness_centrality_retweets.items(), key = lambda item: item[1], reverse = True)[:2]:
    print(f"Node: {node}, Betweenness Centrality: {centrality:.4f}")

print('')
# Calculate Clustering Coefficient
clustering_coefficient_retweets = nx.clustering(G_retweets)
print('Top 2 Clustering Coefficient (Retweets)')

# Filter out nodes with 0 clustering coefficient
meaningful_clustering_retweets = {node: coeff for node, coeff in clustering_coefficient_retweets.items() if coeff > 0}
for node, coeff in sorted(meaningful_clustering_retweets.items(), key = lambda item: item[1], reverse = True)[:2]:
    print(f"Node: {node}, Clustering Coefficient: {coeff:.4f}")

Top 2 Degree Centrality (Retweets)
Node: IAMRAPCHA, Degree Centrality: 0.0553
Node: itskipronoh, Degree Centrality: 0.0454

Top 2 Betweenness (Retweets)
Node: kamwari_, Betweenness Centrality: 0.0002
Node: Ngartia, Betweenness Centrality: 0.0002

Top 2 Clustering Coefficient (Retweets)
Node: chapatizimeisha, Clustering Coefficient: 0.5000
Node: LeakeyKayvoh, Clustering Coefficient: 0.5000
CPU times: user 7min 37s, sys: 458 ms, total: 7min 38s
Wall time: 7min 54s


# **Question Two**